setup

In [1]:
import os
import sys
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from ipywidgets import interact
from jaxtyping import Bool, Float
from torch import Tensor
from tqdm import tqdm

chapter = "chapter0_fundamentals"
section = "part1_ray_tracing"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_ray_tracing.tests as tests
from part1_ray_tracing.utils import (
    render_lines_with_plotly,
    setup_widget_fig_ray,
    setup_widget_fig_triangle
)
from plotly_utils import imshow

MAIN = __name__ == "__main__"

## 1. Ray & Segments

### Exercise - implement `make_rays_1d`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Implement the following `make_rays_1d` function so it generates some rays coming out of the origin, which we'll take to be `(0, 0, 0)`.

Calling `render_lines_with_plotly` on your rays will display them in a 3D plot.

In [2]:
def make_rays_1d(num_pixels: int, y_limit: float) -> Tensor:
    """
    num_pixels: The number of pixels in the y dimension. Since there is one ray per pixel, this is
        also the number of rays.
    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both endpoints.
    
    Returns: shape (num_pixels, num_points=2, num_dims=3) where the num_points dimension contains
        (origin, direction) and the num_dim dimension contains xyz."""
    # create tensor to hold rays
    rays = t.zeros(num_pixels, 2, 3)
    rays[:, 1, 0] = 1 # x = 1 for all rays
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays

```
ray
 ├─ point (origin or direction)
 │   └─ coordinates (x, y, z)
```

In [3]:
rays1d = make_rays_1d(9, 10.0)
fig = render_lines_with_plotly(rays1d.numpy())

**Ray-Object Intersection**

A ray is defined by an **origin** $O$ and a **direction** $D$. Every point along the ray can be written as:

$$R(u) = O + uD$$

where:
- $u \geq 0$
- $u = 0$ gives the origin
- increasing $u$ moves forward along the ray

A line segment is defined by endpoints $L_1$ and $L_2$, and parameterized as:

$$Q(v) = L_1 + v(L_2 - L_1)$$

where:
- $v \in [0, 1]$
- $v = 0$ gives $L_1$
- $v = 1$ gives $L_2$
- $0 < v < 1$ gives a point strictly between them


The ray intersects the segment when both equations describe the same point:

$$O + uD = L_1 + v(L_2 - L_1)$$

That is, walking $u$ units along the ray and $v$ units along the segment both land at the same location.

Rearranging:

$$uD - v(L_2 - L_1) = L_1 - O$$

This is a linear system of the form:

$$A \begin{pmatrix} u \\ v \end{pmatrix} = b$$

Solving gives us values for $u$ and $v$.

Once we have $u$ and $v$, we check two constraints:

1. **Ray constraint:** $u \geq 0$ — the ray only goes forward
2. **Segment constraint:** $0 \leq v \leq 1$ — the hit point must lie between $L_1$ and $L_2$

> **Special case:** If the system has no solution, the ray and segment are parallel — no intersection exists.

Parameterize both objects:

$$R(u) = O + uD \qquad Q(v) = L_1 + v(L_2 - L_1)$$

Then solve for $u$ and $v$ such that $R(u) = Q(v)$, and verify:

$$u \geq 0 \qquad 0 \leq v \leq 1$$

If both conditions hold, the ray intersects the segment.

In [4]:
fig: go.FigureWidget = setup_widget_fig_ray()
display(fig)

@interact(v=(0.0, 6.0, 0.01), seed=(0, 10, 1))
def update(v=0.0, seed=0):
    t.manual_seed(seed)
    L_1, L_2 = t.rand(2, 2)
    P = lambda v: L_1 + v * (L_2 - L_1)
    x, y = zip(P(0), P(6))
    x = [a.item() for a in x]
    y = [b.item() for b in y]
    with fig.batch_update():
        fig.update_traces({"x": x, "y": y}, 0)
        fig.update_traces({"x": [L_1[0].item(), L_2[0].item()], "y": [L_1[1].item(), L_2[1].item()]}, 1)
        fig.update_traces({"x": [P(v)[0].item()], "y": [P(v)[1].item()]}, 2)

FigureWidget({
    'data': [{'type': 'scatter', 'uid': '8842caf2-574a-4c3e-b521-94c4618ea988', 'x': [], 'y': []},
             {'marker': {'size': 12},
              'mode': 'markers',
              'name': 'v=0',
              'type': 'scatter',
              'uid': 'ec89550d-f459-41b6-8498-a79dd19165c9',
              'x': [],
              'y': []},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'name': 'v=1',
              'type': 'scatter',
              'uid': 'd089b4ba-e257-42b4-86a7-adf9a67d8c84',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Ray coordinates illustration'},
               'width': 500,
               'xaxis': {'range': [-1.5, 2.5]},
               'yaxis': {'range': [-1.5, 2.5]}}
})

interactive(children=(FloatSlider(value=0.0, description='v', max=6.0, step=0.01), IntSlider(value=0, descript…

### Exercise - implement `intersect_ray_1d`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-25 minutes on this exercise.
> It involves some of today's core concepts - tensor manipulation, linear operations, etc.
> ```

Using [`torch.linalg.solve`](https://pytorch.org/docs/stable/generated/torch.linalg.solve.html) and [`torch.stack`](https://pytorch.org/docs/stable/generated/torch.stack.html), implement the `intersect_ray_1d` function to solve the above matrix equation.

> yes, if A or the left side of the matrix is not invertable? or if the lines are parallell?

In [5]:
def intersect_ray_1d(ray: Float[Tensor, "points dims"], segment: Float[Tensor, "points dims"]) -> bool:
    """
    ray: shape(n_points=2, n_dim=3) # O, D points
    segment: shape (n_points=2, n_dim=3) # L_1, L_2 points
    
    Return True if the ray intersects the segment.
    """
    r1 = t.tensor([ray[1][0], segment[0][0] - segment[1][0]]) # Dx (L1 - L2)x
    r2 = t.tensor([ray[1][1], segment[0][1] - segment[1][1]]) # Dy (L1 - L2)y
    A = t.stack([r1,r2])
    b = t.tensor([
        (segment[0][0] - ray[0][0]),
        (segment[0][1] - ray[0][1])
    ])
    try:
        u, v = t.linalg.solve(A,b)
    except RuntimeError:
        return False
    return (u >= 0.0 and v >= 0.0 and v <= 1.0)
    

In [ ]:
tests.test_intersect_ray_1d(intersect_ray_1d)
tests.test_intersect_ray_1d_special_case(intersect_ray_1d)

All tests in `test_intersect_ray_1d` passed!
All tests in `test_intersect_ray_1d_special_case` passed!


## 2. Batched Operations

**Batched Ray-Segment Intersection**

### Exercise - implement `intersect_rays_1d`

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 25-40 minutes on this exercise.
> This will probably be one of the hardest exercises you'll complete today.
> ```

One part you might find difficult is dealing with the zero determinant cases. Previously we dealt with those by using `try / except`, but here we can't do that because we want to perform all the operations at once. Instead, we can use the following clever trick to find all the pairs of intersecting rays and segments:

1. Figure out which matrices have zero determinant (e.g. with `determinants.abs() < 1e-8`)
2. Replace those matrices with the identity matrix `t.eye(2)`, since this will certainly not raise an error when solving
3. Find the matrices s.t. `u, v` are in the required range **and** our original matrix was non-singular

This way, we've identified all pairs of rays and segments where an intersection point exists **and** that intersection point is valid (i.e. it's actually on the positive side of the ray, and somewhere in the middle of the segment).

Once we have this 2D array of booleans representing whether each ray intersects with each segment, we can reduce using the torch function `t.any` to find the rays which intersect *any* segment.

Notes:
- `t.linalg.solve()` requires the matrix to be invertable. So, if a matrix determinant is zero, the matrix is not invertible. 
    -  Does that mean the segment and ray are parallel?
    
    The determinant of two vectors equals the area of the parallelogram they form. If vectors are parallel, the parallelogram collapses to a line (Area = 0).

    When directions are parallel there's either:
    - No intersection
    - or, infinite intersections (overlapping)

    Because there isn't one unique intersection point, the linear solver fails.


$$
\begin{aligned}O + u D &= L_1 + v(L_2 - L_1) \\ u D - v(L_2 - L_1) &= L_1 - O  \\ \begin{pmatrix} D_x & (L_1 - L_2)_x \\ D_y & (L_1 - L_2)_y \\ \end{pmatrix} \begin{pmatrix} u \\ v \\ \end{pmatrix} &=  \begin{pmatrix} (L_1 - O)_x \\ (L_1 - O)_y \\ \end{pmatrix} \end{aligned}
$$

Once we've found values of $u$ and $v$ which satisfy this equation, if any (the lines could be parallel) we just need to check that $u \geq 0$ and $v \in [0, 1]$.

In [7]:
def intersect_rays_1d(
        rays: Float[Tensor, "nrays 2 3"], segments: Float[Tensor, "nsegments 2 3"]
) -> Bool[Tensor, " nrays"]:
    """
    For each ray, return True if it intersects any segment.
    """
    device = "mps" if t.backends.mps.is_available() else "cpu"
    rays = rays.to(device)
    segments = segments.to(device)

    # ignore z dim for now
    rays = rays[:, :, :2]
    segments = segments[:, :, :2]

    # we want every ray compared with every segment
    rays = einops.repeat(rays, "nrays pts dims -> nrays nsegments pts dims", nsegments=segments.size(0))
    segments = einops.repeat(segments, "nsegments pts dims -> nrays nsegments pts dims", nrays=rays.size(0))

    # build the linear system
    # rays (nrays, nsegmets, [Origin, Direction], [x,y])
    O = rays[:, :, 0, :] # (nrays, nsegmets 2)
    D = rays[:, :, 1, :] # (nrays, nsegmets 2)
    # segments (nrays, nsegmets, [L_1, L_2], [x,y])
    L_1 = segments[:, :, 0, :] # (nrays, nsegmets 2)
    L_2 = segments[:, :, 1, :] # (nrays, nsegmets 2)
    
    mats = t.stack([D, L_1 - L_2], dim=-1)
    vecs = L_1 - O 

    # chech determinant of the matrix
    determinants = t.linalg.det(mats)
    is_singular = determinants.abs() < 1e-8
    mats[is_singular] = t.eye(2, device=mats.device)

    # solve
    sol= t.linalg.solve(mats, vecs)
    u = sol[..., 0]
    v = sol[..., 1]

    return ((u >= 0.0) & (v >= 0.0) & (v <= 1.0) & ~is_singular).any(dim=-1)


In [8]:
tests.test_intersect_rays_1d(intersect_rays_1d)
tests.test_intersect_rays_1d_special_case(intersect_rays_1d)

All tests in `test_intersect_rays_1d` passed!
All tests in `test_intersect_rays_1d_special_case` passed!


**2D Rays**

### Exercise - implement `make_rays_2d`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

Implement `make_rays_2d` analogously to `make_rays_1d`. The result should look like a pyramid with the tip at the origin.

In [9]:
def make_rays_2d(num_pixels_y: int, num_pixels_z: int, y_limit: float, z_limit: float) -> Float[Tensor, "nrays 2 3"]:
    """
    num_pixels_y: The number of pixels in the y dimension
    num_pixels_z: The number of pixels in the z dimension
    
    y_limit: AT x=1, the rays should extend from -y_limit to +y_limit, inclusive of both.
    z_limit: AT x=1, the rays should extend from -z_limit to +z_limit, inclusive of both.
    
    Returns: shape (num_rays=num_pixels_y * num_pixels_z, num_points=2, num_dims=3).
    """
    rays = t.zeros(num_pixels_y*num_pixels_z, 2, 3)
    # for the second point (direction) we want x = 1
    rays[:, 1, 0] = 1
    y = t.linspace(-y_limit, y_limit, num_pixels_y)
    z = t.linspace(-z_limit, z_limit, num_pixels_z)
    y = einops.repeat(y, "y_vals -> y_vals num_pixels_z", num_pixels_z=num_pixels_z).flatten()
    z = einops.repeat(z, "z_vals -> num_pixels_y z_vals", num_pixels_y=num_pixels_y).flatten()
    rays[:, 1, 1] = y
    rays[:, 1, 2] = z
    return rays

In [10]:
rays_2d = make_rays_2d(10, 10, 0.3, 0.3)
render_lines_with_plotly(rays_2d.numpy())

## 3. Triangles

### Triangle Coordinates

The area inside a triangle can be defined by three (non-collinear) points $A$, $B$ and $C$, and can be written algebraically as a **convex combination** of those three points:

$$
\begin{align*}
P(w, u, v) &= wA + uB + vC \quad\quad \\
    \\
s.t. \quad 0 &\leq w,u,v \\
1 &= w + u + v
\end{align*}
$$

Or equivalently:

$$
\begin{align*}
\quad\quad\quad\quad P(u, v) &= (1 - u - v)A + uB + vC \\
&= A + u(B - A) + v(C - A) \\
\\
s.t. \quad 0 &\leq u,v \\
u + v &\leq 1
\end{align*}
$$

These $u, v$ are called "barycentric coordinates".

If we remove the bounds on $u$ and $v$, we get an equation for the plane containing the triangle. Play with the widget to understand the behavior of $u, v$.

Notes:
- Any point inside the triangle can be written as a weighted average of these three points (A, B, C).
- convex combination means a weighted average where the weights are non-negateve and sum to 1?
- $A + u(B - A) + v(C - A)$ this alternative form can be interpreted as:
    - start at A
    - then move u amount toward B ($u(B - A)$)
    - move v amount toward C ($v(C - A)$)

In [11]:
one_triangle = t.tensor([[0, 0, 0], [4, 0.5, 0], [2, 3, 0]])
A, B, C = one_triangle
x, y, z = one_triangle.T

fig: go.FigureWidget = setup_widget_fig_triangle(x.numpy(), y.numpy(), z.numpy())
display(fig)

@interact(u=(-0.5, 1.5, 0.01), v=(-0.5, 1.5, 0.01))
def update(u=0.0, v=0.0):
    P = A + u * (B - A) + v * (C - A)
    fig.update_traces({"x": [P[0].item()], "y": [P[1].item()]}, 2)

FigureWidget({
    'data': [{'marker': {'size': 12},
              'mode': 'markers+text',
              'text': [A, B, C],
              'textfont': {'size': 18},
              'textposition': 'middle left',
              'type': 'scatter',
              'uid': 'da9378a0-2d12-4fab-a639-2f488d869485',
              'x': {'bdata': 'AAAAAAAAgEAAAABA', 'dtype': 'f4'},
              'y': {'bdata': 'AAAAAAAAAD8AAEBA', 'dtype': 'f4'}},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': '9a134769-1c3b-43ec-8b8d-442d48e5b655',
              'x': [0.0, 4.0, 2.0, 0.0],
              'y': [0.0, 0.5, 3.0, 0.0]},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'type': 'scatter',
              'uid': '3cd1473d-8b0b-4670-9378-4efcf0aaf3d1',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
          

interactive(children=(FloatSlider(value=0.0, description='u', max=1.5, min=-0.5, step=0.01), FloatSlider(value…

### Triangle-Ray Intersection

Given a ray with origin $O$ and direction $D$, our intersection algorithm will consist of two steps:

- Finding the intersection between the line and the plane containing the triangle, by solving the equation $P(u, v) = P(s)$;
- Checking if $u$ and $v$ are within the bounds of the triangle.

Expanding the equation $P(u, v) = P(s)$, we have:

$$
\begin{align*}
A + u(B - A) + v(C - A) &= O + sD \\
\Rightarrow
\begin{pmatrix}
    -D & (B - A) & (C - A) \\
\end{pmatrix}
\begin{pmatrix}
    s \\
    u \\
    v  
\end{pmatrix}
&= \begin{pmatrix} O - A \end{pmatrix} \\
\Rightarrow \begin{pmatrix}
    -D_x & (B - A)_x & (C - A)_x \\
    -D_y & (B - A)_y & (C - A)_y \\
    -D_z & (B - A)_z & (C - A)_z \\
\end{pmatrix}
\begin{pmatrix}
    s \\
    u \\
    v  
\end{pmatrix} &= \begin{pmatrix}
    (O - A)_x \\
    (O - A)_y \\
    (O - A)_z \\
\end{pmatrix}
\end{align*}
$$

$$
$$

We can therefore find the coordinates `s`, `u`, `v` of the intersection point by solving the linear system above.

Notes:
$$
\begin{aligned}
A + u(B - A) + v(C - A) &= O + sD \\
A + u(B - A) + v(C - A) - A &= O + sD - A \\
u(B - A) + v(C - A) &= O + sD - A \\
u(B - A) + v(C - A) - sD &= O - A \\
-sD + u(B - A) + v(C - A) &= O - A \\
\Rightarrow
\begin{bmatrix}
-D & (B - A) & (C - A)
\end{bmatrix}
\begin{bmatrix}
s \\
u \\
v
\end{bmatrix}
&=
\begin{bmatrix}
O - A
\end{bmatrix} \\
\Rightarrow
\begin{bmatrix}
-D_x & (B - A)_x & (C - A)_x \\
-D_y & (B - A)_y & (C - A)_y \\
-D_z & (B - A)_z & (C - A)_z
\end{bmatrix}
\begin{bmatrix}
s \\
u \\
v
\end{bmatrix}
&=
\begin{bmatrix}
(O - A)_x \\
(O - A)_y \\
(O - A)_z
\end{bmatrix}
\end{aligned}
$$
- A ray starts at a point and goes in a direction $P(s) = O + sD$
- if the ray hits the triangle, there must exist a point that staisfies both equations $P(u, v) = P(s)$
- we want to solve for:
    - s: where along the ray
    - u: triangle barycentric coordinate
    - v: triangle barycentric coordinate
- The condition to check
    - s >= 0
    - u >= 0
    - v >= 0
    - u + v <= 1 

### Exercise - implement `triangle_ray_intersects`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

Using `torch.linalg.solve` and `torch.stack`, implement `triangle_ray_intersects(A, B, C, O, D)`.

A few tips:

* If you have a 0-dimensional tensor with shape `()` containing a single value, use the `item()` method to convert it to a plain Python value.
* If you have a tensor of shape `tensor.shape = (3, ...)`, then you can unpack it along the first dimension into three separate tensors the same way that you'd unpack a normal python list: `s, u, v = tensor`.
    * Note - if the dimension you want to unpack isn't at the start, a nice alternative is `s, u, v = tensor.unbind(dim)`, which does the same thing but along the dimension given by `dim` rather than the first dimension.
* If your function isn't working, try making a simple ray and triangle with nice round numbers where you can work out manually if it should intersect or not, then debug from there.

In [12]:
A = t.tensor([0., 0., 0.])
B = t.tensor([2., 0., 0.])
C = t.tensor([0., 2., 0.])

O = t.tensor([0.5, 0.5, 1.])
D = t.tensor([0., 0., -1.])

In [13]:
mat = t.stack([-D, B - A, C - A], dim=-1)
vec = O - A
s, u, v = t.linalg.solve(mat, vec)
s.item() >= 0 and u.item() >= 0 and v.item() >= 0 and u.item() + v.item() <=1

True

In [14]:
Point = Float[Tensor, "points=3"]

def triangle_ray_intersects(A: Point, B: Point, C: Point, O: Point, D: Point) -> bool:
    """
    A: shape (3,), one vertex of the triangle
    B: shape (3,), second vertex of the triangle
    C: shape (3,), third vertex of the triangle
    O: shape (3,), origin point
    D: shape (3,), direction vector

    return True if the ray and the triangle intersect.
    """
    # setup the matrix and vector for: matrix x unknowns = vector
    mat = t.stack([-D, B - A, C - A], dim=-1)
    vec = O - A
    s, u, v = t.linalg.solve(mat, vec)
    return s.item() >= 0 and u.item() >= 0 and v.item() >= 0 and u.item() + v.item() <=1

In [15]:
tests.test_triangle_ray_intersects(triangle_ray_intersects)

All tests in `test_triangle_ray_intersects` passed!


### Single-Triangle Rendering

Implement `raytrace_triangle` using only one call to `torch.linalg.solve`.

Reshape the output and visualize with `plt.imshow`. It's normal for the edges to look pixelated and jagged - using a small number of pixels is a good way to debug quickly.

If you think it's working, increase the number of pixels and verify that it looks less pixelated at higher resolution.

### Exercise - implement `raytrace_triangle`

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> This is about as hard as `intersect_rays_1d`, although hopefully you should find it more familiar.
> ```

Below, you should implement `raytrace_triangle`, a funtion which checks whether each ray in `ray` intersects a single given triangle.

rays
```
|--nrays
     |--- Origin
            |----- x, y, z
     |--- Direction
            |----- x, y, z
```

In [16]:
def raytrace_triangle(
    rays: Float[Tensor, "nrays rayPoints=2 dims=3"],
    triangle: Float[Tensor, "trianglePoints=3 dims=3"]
) -> Bool[Tensor, "nrays"]:
    """
    For each ray, return True if the triangle intersects that ray.
    """
    # first, I need to extract the tensors I need for the linear equation
    O = rays[:, 0]
    D = rays[:, 1]
    A = triangle[0]
    B = triangle[1]
    C = triangle[2]
    # I think I need to repeat each point (A, B, C) nray times to be able to setup the matrix for each ray
    A = einops.repeat(A, "points -> nrays points", nrays=rays.size(0))
    B = einops.repeat(B, "points -> nrays points", nrays=rays.size(0))
    C = einops.repeat(C, "points -> nrays points", nrays=rays.size(0))
    # now I can set up matrix and vector to solve matrix x unkown = vector
    # this will be multi matrix tensor. a matrix for each ray
    mat = t.stack([-D, B - A, C - A], dim=-1)
    vec = O - A
    # handle singular matrices
    dets = t.linalg.det(mat)
    is_singular = dets.abs() < 1e-8
    mat[is_singular] = t.eye(3)
    # now we can solve
    sol = t.linalg.solve(mat, vec)
    s = sol[..., 0]
    u = sol[..., 1]
    v = sol[..., 2]
    # check conditions
    return (s >= 0) & (u >= 0) & (v >= 0) & (u + v <= 1) & ~is_singular

In [17]:
A = t.tensor([1, 0.0, -0.5])
B = t.tensor([1, -0.5, 0.0])
C = t.tensor([1, 0.5, 0.5])
num_pixels_y = num_pixels_z = 60
y_limit = z_limit = 0.5

# Plot triangle and rays
test_triangle = t.stack([A, B, C], dim=0)
rays2d = make_rays_2d(num_pixels_y, num_pixels_z, y_limit, z_limit)
triangle_lines = t.stack([A, B, C, A, B, C], dim=0). reshape(-1, 2, 3)
render_lines_with_plotly(rays2d.numpy(), triangle_lines.numpy())

In [18]:
# Calculate and display intersections
intersects = raytrace_triangle(rays2d, test_triangle)
img = intersects.reshape(num_pixels_y, num_pixels_z).int()
imshow(img, origin="lower", width=600, title="Triangle (as intersected by rays)")

### Debugging

Debugging code is an extremely important thing to learn. Just like using GPT to assist your coding, it's something that can significantly speedrun your development, and stop you getting hung up on all the frustrating things!

To give you practice debugging using VSCode's built-in debugger\*, we've provided an example function below. This is an implementation of `raytrace_triangle` which has a few mistakes in it. Your task is to use the debugger to find the mistake and fix it. (Note - you're encouraged to actually use the debugger, rather than comparing it to the solution above!)

### Incorrect function (I debugged using the debugger + found the issues and fixed them)

In [19]:
def raytrace_triangle_with_bug(
    rays: Float[Tensor, "nrays rayPoints=2 dims=3"],
    triangle: Float[Tensor, "trianglePoints=3 dims=3"]
) -> Bool[Tensor, " nrays"]:
    '''
    For each ray, return True if the triangle intersects that ray.
    '''
    NR = rays.size(0)

    A, B, C = einops.repeat(triangle, "pts dims -> pts NR dims", NR=NR)

    O, D = rays.unbind(dim=1)

    mat = t.stack([- D, B - A, C - A], dim=-1)
    
    dets = t.linalg.det(mat)
    is_singular = dets.abs() < 1e-8
    mat[is_singular] = t.eye(3)

    vec = O - A

    sol = t.linalg.solve(mat, vec)
    s, u, v = sol.unbind(dim=-1)

    return ((u >= 0) & (v >= 0) & (u + v <= 1) & ~is_singular)


intersects = raytrace_triangle_with_bug(rays2d, test_triangle)
img = intersects.reshape(num_pixels_y, num_pixels_z).int()
imshow(img, origin="lower", width=600, title="Triangle (as intersected by rays)")

You can debug a cell by clicking on the **Debug cell** option at the bottom of your cell. Your cell should contain the code that is actually run to cause an error (rather than needing to contain the function which is the source of the error). Before you run your debugger, you can set breakpoints by clicking on the left-hand side of the line number (a red dot will appear). You can then step through your code, using the toolbar of buttons which will appear when you run the debugger (see [here](https://pawelgrzybek.com/continue-step-over-step-into-and-step-out-actions-in-visual-studio-code-debugger-explained/) for an explanation of what each of the buttons does). When you reach a breakpoint, you can use the following tools:

* Inspect local and global variables, in the **VARIABLES** window that appears on the left sidebar.
* Add variables expressions to watch, in the **WATCH** window that appears on the left sidebar.
    * You can just type in any expression here, e.g. the type of a variable or the length of a list, and this will update as you step through the function.
* Evaluate expressions on a one-time basis, by typing them into the **DEBUG CONSOLE** (which appears at the bottom of the screen)

Note, when you run the debugger, it will stop *before* the breakpoint line is evaluated. So if you've run a cell and got an error on a specific line, then *that* is the line you want to set a breakpoint on.

*This all works basically the same if you're in a notebook in VSCode, except for a few changes e.g. the debug button is on the dropdown at the top-left of the cell (if it doesn't appear for you then you'll need to go into user settings and add the line `"notebook.consolidatedRunButton": true`).*

There's much more detail we could go into about debugging, but this will suffice for most purposes. It's generally a much more efficient way of debugging than using print statements or asserts (although these can also be helpful in some situations).

### Mesh Loading

In [20]:
triangles = t.load(section_dir / "pikachu.pt", weights_only=True)

In [21]:
triangles.shape

torch.Size([412, 3, 3])

### Mesh Rendering

For our purposes, a mesh is just a group of triangles, so to render it we'll intersect all rays and all triangles at once. We previously just returned a boolean for whether a given ray intersects the triangle, but now it's possible that more than one triangle intersects a given ray.

For each ray (pixel) we will return a float representing the minimum distance to a triangle if applicable, otherwise the special value `float('inf')` representing infinity. We won't return which triangle was intersected for now.

Note - by distance to a triangle, we specifically mean **distance along the x-dimension**, not Euclidean distance.

### Exercise - implement `raytrace_mesh`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-25 minutes on this exercise.
> 
> This is the main function we've been building towards, and marks the end of the core exercises. If should involve a lot of repurposed code from the last excercise.
> ```

Implement `raytrace_mesh` and as before, reshape and visualize the output. Your Pikachu is centered on (0, 0, 0), so you'll want to slide the ray origin back to at least `x=-2` to see it properly.

Reminder - `t.linalg.solve` (and most batched operations) can accept multiple dimensions as being batch dims. Previously, you've just used `NR` (the number of rays) as the batch dimension, but you can also use `(NR, NT)` (the number of rays and triangles) as your batch dimensions, so you can solve for all rays and triangles at once.

rays
```
|--nrays
     |--- Origin
            |----- x, y, z
     |--- Direction
            |----- x, y, z
```

triangles
```
|--ntriangles
       |--- A
            |----- x, y, z
       |--- B
            |----- x, y, z
       |--- C
            |----- x, y, z
```

In [23]:
def raytrace_mesh(
        rays: Float[Tensor, "nrays rayPoints=2 dims=3"],
        triangles: Float[Tensor, "ntriangles trianglePoints= 3 dims=3"],
) -> Float[Tensor, " nrays"]:
    """
    For each ray, return the distance to the closest intersecting triangle, or infinity.
    """
    NR = rays.size(0)
    NT = triangles.size(0)
    # extract O, D, A, B, C
    O = rays[:, 0] # we shift this to x=-2 later so I don't think we need to do it here
    D = rays[:, 1]
    A = triangles[:, 0]
    B = triangles[:, 1]
    C = triangles[:, 2]
    # To render, we need to intersect all rays and all triangles at once
    # For each ray (pixel), we need to first check if we're hitting a triangle
    # I need to make the shapes broadcastable between the ray and triangle points
    O = einops.repeat(O, "nr pts -> nr nt pts", nt=NT)
    D = einops.repeat(D, "nr pts -> nr nt pts", nt=NT)
    A = einops.repeat(A, "nt pts -> nr nt pts", nr=NR)
    B = einops.repeat(B, "nt pts -> nr nt pts", nr=NR)
    C = einops.repeat(C, "nt pts -> nr nt pts", nr=NR)
    # now I can set up mat and vec for solving the linear equation
    mat = t.stack([-D, B - A, C - A], dim=-1)
    vec = O - A
    # handle singulare (non-invertible matrices)
    dets = t.linalg.det(mat)
    is_singular = dets.abs() < 1e-8
    mat[is_singular] = t.eye(3)
    # solve
    sol = t.linalg.solve(mat, vec)
    # extract solution vectors
    s = sol[..., 0]
    u = sol[..., 1]
    v = sol[..., 2]
    # now I can check the condition to see if a ray intersects with a triangle
    interacts = ((s >= 0) & (u >= 0) & (v >= 0) & (u + v <= 1) & ~is_singular)
    # s is how far along the dirction vector are we when we hit an onject
    out = t.where(interacts, s, float('inf'))
    return out.min(dim=-1).values

In [24]:
num_pixels_y = 120
num_pixels_z = 120
y_limit = z_limit = 1

rays = make_rays_2d(num_pixels_y, num_pixels_z, y_limit, z_limit)
rays[:, 0] = t.tensor([-2, 0.0, 0.0])
dists = raytrace_mesh(rays, triangles)
intersects = t.isfinite(dists).view(num_pixels_y, num_pixels_z)
dists_square = dists.view(num_pixels_y, num_pixels_z)
img = t.stack([intersects, dists_square], dim=0)

fig = px.imshow(img, facet_col=0, origin="lower", color_continuous_scale="magma", width=1000)
fig.update_layout(coloraxis_showscale=False)
for i, text, in enumerate(["intersects", "Distance"]):
    fig.layout.annotations[i]["text"] = text
fig.show()